<a href="https://colab.research.google.com/github/sanctuary-architect/Tensorscript/blob/main/TensorScript%20T4%20Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TensorScript → T4 (Colab)

Run the full SFT + DPO pipeline from the TensorScript v1.0 spec, compiled to
real `transformers`/`peft` code, on a Colab T4 (16 GB). Uses QLoRA (4-bit base),
fp16, gradient checkpointing — the `hardware: {profile: "t4"}` block drives all
of that automatically.

**Runtime:** T4 GPU · High-RAM


In [37]:
!pip install -U transformers

## Local Inference on GPU
Model page: https://huggingface.co/meta-llama/Meta-Llama-3-8B

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/meta-llama/Meta-Llama-3-8B)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

The model you are trying to use is gated. Please make sure you have access to it by visiting the model page.To run inference, either set HF_TOKEN in your environment variables/ Secrets or run the following cell to login. 🤗

In [36]:
from huggingface_hub import login
login()

In [35]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-generation", model="meta-llama/Meta-Llama-3-8B")

SystemError: <class 'numpy.iinfo'> returned a result with an exception set

In [33]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Meta-Llama-3-8B", device_map="auto")

SystemError: <class 'numpy.iinfo'> returned a result with an exception set

In [21]:
# Cell 1 — deps (run once)
!pip -q install -U transformers peft accelerate datasets trl bitsandbytes wandb
!nvidia-smi


/bin/bash: line 1: nvidia-smi: command not found


## Cell 2 — spec

This is the TensorScript spec. The transpiler turns it into Python you could
write by hand — but with the guardrails, telemetry, pipeline wiring, and
hardware profile baked in.


In [52]:
%%writefile align_example.tensor
tensorscript v1.0

dataset SFTData streams {
    source: "hf://datasets/HuggingFaceH4/ultrachat_200k",
    tokenize: AutoTokenizer("gpt2"),
    sequence_length: 1024,
    split: [train_sft: 99%, val_sft: 1%]
}

model BaseLM {
    base: "hf://gpt2",
    quantize: 4bit,
    peft: LoRA(r=16, alpha=32, target_modules="all-linear")
}

monitor RunTelemetry {
    track: [loss, perplexity, memory_usage, reward_margin, kl_divergence],
    checkpoint_every: 500.steps,
    select_best_on: loss minimize,
    destination: "wandb://pioneer-space/sft-dpo-pipeline"
}

optimize BaseLM as SFTStage {
    using: dataset.SFTData.train_sft,
    hardware: {profile: "t4"},
    telemetry: monitor.RunTelemetry,
    schedule: CosineAnnealing(max_lr=2e-4, min_lr=2e-5, warmup=0.03),
    epochs: 1,
    guardrails: [
        if loss > 2.5: rollback_and_scale_lr(0.5),
        if loss flat_lines(epsilon=1e-4) for 200.steps: terminate_early
    ]
}

dataset PreferenceData streams {
    source: "hf://datasets/Anthropic/hh-rlhf",
    tokenize: AutoTokenizer("gpt2"),
    sequence_length: 1024,
    split: [train: 90%, val: 10%]
}

model DPOModel {
    base: pipeline.SFTStage.best,
    quantize: 4bit,
    peft: LoRA(r=16, alpha=32, target_modules="all-linear")
}

optimize DPOModel as DPOStage {
    using: dataset.PreferenceData.train,
    hardware: {profile: "t4"},
    telemetry: monitor.RunTelemetry,
    schedule: CosineAnnealing(max_lr=5e-6, min_lr=5e-7, warmup=0.1),
    epochs: 1,
    guardrails: [
        if reward_margin < 0: rollback_and_scale_lr(0.3),
        if kl_divergence > 10: rollback_and_scale_lr(0.5)
    ]
}

pipeline AlignmentRun {
    stage one {
        run: optimize.SFTStage
    },
    stage two {
        run: optimize.DPOStage,
        depends_on: one,
        inherit_weights: best
    }
}


Overwriting align_example.tensor


## Cell 3 — transpile (auto-downloads the toolchain)

The toolchain is fetched straight from your GitHub repo, then validated, then
the transpiler emits `SFTStage.py`, `DPOStage.py`, and `run_AlignmentRun.py`.
No manual uploads needed.


In [53]:
import sys, os, py_compile
import re

BASE = "https://raw.githubusercontent.com/sanctuary-architect/Tensorscript/main/"
files_to_download = ["parser.py", "transpiler.py", "semantic_checks.py", "tsc_check.py", "lexer.py"]

print("Ensuring toolchain files are present...")
for f in files_to_download:
    if not os.path.exists(f):
        print(f"Downloading {f}...")
        !wget -q $BASE$f

for f in ["parser.py", "transpiler.py", "semantic_checks.py", "lexer.py"]:
    py_compile.compile(f, doraise=True)
print("Toolchain ready")

import sys; sys.path.insert(0, ".")
from semantic_checks import check
from parser import parse
from transpiler import transpile

ast = parse(open("align_example.tensor").read())
check(ast)
out = transpile(ast, out_dir=".")

for filename, content in out.items():
    if filename in ["SFTStage.py", "DPOStage.py"]:
        updated_content = content

        if "import wandb" in updated_content and "wandb.init()" not in updated_content:
            updated_content = updated_content.replace("import wandb", "import wandb\n    wandb.init()")

        if filename == "SFTStage.py":
            # Updated regex to match the specific structure found in the check
            old_tokenize_fn_pattern = re.compile(
                r'def _tokenize_fn\(batch\):\n'
                r'    out = tokenizer\(batch\.get\("text", batch\.get\("chosen", \[""\]\)\), truncation=True, max_length=SEQUENCE_LENGTH, padding="max_length"\)\n'
                r'    out\["labels"\] = out\["input_ids"\]\.copy\(\)\n'
                r'    return out'
            )

            new_tokenize_fn_code = """def _tokenize_fn(batch):
    formatted_texts = []
    for messages_list in batch["messages"]:
        full_text = " ".join([f"{m['role']}: {m['content']}" for m in messages_list])
        formatted_texts.append(full_text)
    out = tokenizer(formatted_texts, truncation=True, max_length=SEQUENCE_LENGTH, padding="max_length")
    out["labels"] = out["input_ids"].copy()
    return out"""

            updated_content = old_tokenize_fn_pattern.sub(new_tokenize_fn_code, updated_content)

        final_lines = []
        for line in updated_content.splitlines():
            if "save_steps=" in line: final_lines.append("    save_steps=500,")
            elif "eval_steps=" in line: final_lines.append("    eval_steps=500,")
            elif "split='train'" in line and "ultrachat_200k" in updated_content:
                final_lines.append(line.replace("split='train'", "split='train_sft'"))
            else: final_lines.append(line)

        with open(filename, "w") as f:
            f.write("\n".join(final_lines))
        print(f"Successfully wrote {filename}")
    else:
        with open(filename, "w") as f:
            f.write(content)
        print(f"Successfully wrote {filename}")

Ensuring toolchain files are present...
Toolchain ready
Successfully wrote SFTStage.py
Successfully wrote DPOStage.py
Successfully wrote run_AlignmentRun.py


## Cell 4 — Hugging Face login

`meta-llama/Meta-Llama-3-8B` is a **gated model**: you must accept its license on
huggingface.co (Meta's terms) and paste a read token here.

🔐 **Private-key safety:** this box runs a login *prompt* — your real token is
typed into it at runtime and is **never saved into this notebook file**, so it
can't leak if the notebook is public. Create a read-only token at
huggingface.co/settings/tokens (type: Read).


In [54]:
# Cell 4 — log in to Hugging Face (paste your READ token below)
from huggingface_hub import notebook_login
notebook_login()


## Cell 5 — run the pipeline

SFT first (download + fine-tune), then DPO on the SFT best checkpoint.
`run_AlignmentRun.py` resolves `pipeline.SFTStage.best` automatically.

> ⚠️ `sequence_length: 1024` + batch 1 + gradient checkpointing ≈ 12–14 GB
> VRAM. If you hit OOM, drop to `sequence_length: 512`.


In [ ]:
# Cell 5 — execute (SFT then DPO; ~30–60 min on a T4)
!python run_AlignmentRun.py


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: sanctuary-architect (sanctuary-architect-none) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.29.0
wandb: Run data is saved locally in /content/wandb/run-20260906_214830-vxaoqzv6
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run confused-plant-8
wandb: ⭐️ View project at https://wandb.ai/sanctuary-architect-none/uncategorized
wandb: 🚀 View run at https://wandb.ai/sanctuary-architect-none/uncategorized/runs/vxaoqzv6
[tensorscript] no seed declared — generated SEED=778445984, logging for reproducibility


In [56]:
print('--- SFTStage.py (Model/Tokenizer Section) ---')
with open('SFTStage.py', 'r') as f:
    lines = f.readlines()
    for line in lines:
        if 'AutoTokenizer.from_pretrained' in line or 'AutoModelForCausalLM.from_pretrained' in line:
            print(line.strip())

print('\n--- Full _tokenize_fn in SFTStage.py ---')
inside_fn = False
for line in lines:
    if 'def _tokenize_fn' in line:
        inside_fn = True
    if inside_fn:
        print(line.rstrip())
        if 'return tokenizer' in line:
            inside_fn = False

--- SFTStage.py (Model/Tokenizer Section) ---
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained(_base_path, quantization_config=_bnb_config, device_map='auto', torch_dtype=torch.float16, attn_implementation='eager')

--- Full _tokenize_fn in SFTStage.py ---
def _tokenize_fn(batch):
    formatted_texts = []
    for messages_list in batch["messages"]:
        full_text = " ".join([f"{m['role']}: {m['content']}" for m in messages_list])
        formatted_texts.append(full_text)
    out = tokenizer(formatted_texts, truncation=True, max_length=SEQUENCE_LENGTH, padding="max_length")
    out["labels"] = out["input_ids"].copy()
    return out

if tokenizer is not None:
    train_dataset = train_dataset.map(_tokenize_fn, batched=True)
    if eval_dataset is not None:
        eval_dataset = eval_dataset.map(_tokenize_fn, batched=True)

# ---- model ----
_base_path = "gpt2"
_bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torc

## Cell 6 — inspect the guardrail log + outputs

The console shows `[guardrail]` warnings if a rule fired. The outputs land in
`./output/SFTStage` and `./output/DPOStage`.


In [27]:
# Cell 6 — inspect outputs
import glob, os
for d in ["output/SFTStage", "output/DPOStage"]:
    if os.path.isdir(d):
        print(d, "->", os.listdir(d)[:6])

with open('SFTStage.py', 'r') as f:
    lines = f.readlines()
    for i, line in enumerate(lines):
        if 'TrainingArguments(' in line:
            print("".join(lines[i:i+20]))
            break


training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    learning_rate=0.0002,
    warmup_steps=60,
    lr_scheduler_type='cosine',
    save_strategy="steps",
    save_steps=500,
    report_to=['wandb'],
    seed=SEED,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_checkpointing=True,
    bf16=False,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    eval_strategy="steps",
)

